## Cleaning

### Data Import

In [41]:
import polars as pl
import altair as alt
from dotenv import load_dotenv
import os

load_dotenv()  # automatically finds .env in current or parent directory

USER = os.getenv("USER")
PASSWORD = os.getenv("PW")
HOST = os.getenv("HOST")
DATABASE = os.getenv("DB_NAME")

uri = f"postgresql://{USER}:{PASSWORD}@{HOST}:5432/{DATABASE}"

query = "SELECT * FROM daan_822.article_detailed"

df = pl.read_database_uri(query=query, uri=uri)

In [42]:
df.describe()

statistic,pmid,doi,article_type,article_title,article_subject,authors,pub_date,keywords,reference_count,license_type,journal_title,publisher_name,copyright_statement,copyright_year,abstract,affiliations,has_supplemental,figure_count,table_count,funding,data_available_details,data_available,code_available_details,code_available
str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,f64,str,str,str,str,f64,str,f64
"""count""","""7178""","""7167""","""7178""","""7178""","""7178""","""7178""","""7178""","""7178""",7178.0,"""0""","""7178""","""7178""","""7178""","""7178""","""7178""","""7178""",7178.0,"""7178""","""7178""","""7178""","""7178""",7178.0,"""7178""",7178.0
"""null_count""","""0""","""11""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""7178""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""",0.0,"""0""",0.0
"""mean""",null,null,null,null,null,null,null,null,53.387852,null,null,null,null,null,null,null,0.657147,null,null,null,null,0.395653,null,0.033853
"""std""",null,null,null,null,null,null,null,null,48.141303,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""38168309""","""10.1001/jamahealthforum.2025.1…","""abstract""",""" Helicobacter pylori Screeni…","""""","""[]""","""2024-10-08""","""["" Artificial intelligence"",""S…",0.0,null,"""AACE Endocrinology and Diabete…","""""","""""","""""","""""","""[""*Brain Trauma Foundation, Pa…",0.0,"""0""","""0""","""["" The APC was funded by Wenta…","""["" Data are available on reque…",0.0,"""[""A GitHub repository containi…",0.0
"""25%""",null,null,null,null,null,null,null,null,31.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null,null,43.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null,null,59.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""41716853""","""10.9758/cpn.24.1261""","""systematic-review""","""대형 언어 모델을 활용한 연구를 위한 TRIPOD-LL…","""Zoster""","""[{""given_names"":""Šimon"",""is_co…","""2026-5-12""","""[]""",1194.0,null,"""npj Metabolic Health and Disea…","""ecancer Global Foundation""","""©Zongqi Xia, Prerna Chikersal,…","""2026""","""​​BackgroundOsteoporosis, a pr…","""[]""",1.0,"""9""","""9""","""[]""","""[]""",1.0,"""[]""",1.0


### Cleaning

In [43]:
# cleaning whitespace and replacing blank with true nulls
df_cleaning = df.with_columns(pl.selectors.string().str.strip_chars().replace("", None))

# converting string type to int
df_cleaning = df_cleaning.with_columns(
    pl.selectors.by_name("copyright_year", "figure_count", "table_count").cast(pl.Int16)
)

# converting pub_date to date
df_cleaning = df_cleaning.with_columns(
    pl.coalesce(
        [
            pl.col("pub_date").str.to_date("%Y-%m-%d", strict=False),
            pl.col("pub_date").str.to_date("%Y-%m", strict=False),
            pl.col("pub_date").str.to_date("%Y", strict=False),
        ]
    )
)

# dropping column as it is null
df_cleaning = df_cleaning.drop("license_type")

# making columns lower case
df_cleaning = df_cleaning.with_columns(pl.selectors.by_name("keywords", "journal_title", "publisher_name").str.to_lowercase())

In [44]:
df_cleaning.describe()

statistic,pmid,doi,article_type,article_title,article_subject,authors,pub_date,keywords,reference_count,journal_title,publisher_name,copyright_statement,copyright_year,abstract,affiliations,has_supplemental,figure_count,table_count,funding,data_available_details,data_available,code_available_details,code_available
str,str,str,str,str,str,str,str,str,f64,str,str,str,f64,str,str,f64,f64,f64,str,str,f64,str,f64
"""count""","""7178""","""7167""","""7178""","""7178""","""7140""","""7178""","""7156""","""7178""",7178.0,"""7178""","""6518""","""6871""",6105.0,"""7117""","""7178""",7178.0,7178.0,7178.0,"""7178""","""7178""",7178.0,"""7178""",7178.0
"""null_count""","""0""","""11""","""0""","""0""","""38""","""0""","""22""","""0""",0.0,"""0""","""660""","""307""",1073.0,"""61""","""0""",0.0,0.0,0.0,"""0""","""0""",0.0,"""0""",0.0
"""mean""",null,null,null,null,null,null,"""2025-07-02 23:04:15.561766""",null,53.387852,null,null,null,2024.984767,null,null,0.657147,3.942045,3.011702,null,null,0.395653,null,0.033853
"""std""",null,null,null,null,null,null,null,null,48.141303,null,null,null,0.150133,null,null,null,7.704139,2.888119,null,null,null,null,null
"""min""","""38168309""","""10.1001/jamahealthforum.2025.1…","""abstract""","""10 Practical Considerations fo…","""10""","""[]""","""2024-03-20""","""["" artificial intelligence"",""s…",0.0,"""aace endocrinology and diabete…","""a k peters/crc press""","""(c) 2025 The authors; licensee…",2023.0,"""(1) Background: Alveolar bone …","""[""*Brain Trauma Foundation, Pa…",0.0,0.0,0.0,"""["" The APC was funded by Wenta…","""["" Data are available on reque…",0.0,"""[""A GitHub repository containi…",0.0
"""25%""",null,null,null,null,null,null,"""2025-04-04""",null,31.0,null,null,null,2025.0,null,null,null,2.0,2.0,null,null,null,null,null
"""50%""",null,null,null,null,null,null,"""2025-07-07""",null,43.0,null,null,null,2025.0,null,null,null,3.0,3.0,null,null,null,null,null
"""75%""",null,null,null,null,null,null,"""2025-10-06""",null,59.0,null,null,null,2025.0,null,null,null,5.0,4.0,null,null,null,null,null
"""max""","""41716853""","""10.9758/cpn.24.1261""","""systematic-review""","""대형 언어 모델을 활용한 연구를 위한 TRIPOD-LL…","""Zoster""","""[{""given_names"":""Šimon"",""is_co…","""2026-05-12""","""[]""",1194.0,"""yonsei medical journal""","""zenodo""","""©Zongqi Xia, Prerna Chikersal,…",2026.0,"""​​BackgroundOsteoporosis, a pr…","""[]""",1.0,487.0,77.0,"""[]""","""[]""",1.0,"""[]""",1.0


In [45]:
# labelling field names in the json construct for authors
dtypes = pl.List(
    pl.Struct(
        [
            pl.Field("given_names", pl.String),
            pl.Field("is_corresponding", pl.Boolean),
            pl.Field("orcid", pl.String),
            pl.Field("surname", pl.String),
        ]
    )
)

# add individual fileds for each in the authors json construct
df_parsed = (
    df_cleaning.with_columns(
        pl.col("authors").str.json_decode(dtypes).alias("authors_struct"),
        pl.col("keywords").str.json_decode(pl.List(pl.String)).alias("keywords_list"),
    )
    .explode("authors_struct")
    .unnest("authors_struct")
    .with_columns(
        (pl.col("given_names") + " " + pl.col("surname")).alias("author_full_name")
    )
    .explode("keywords_list")
    .rename({"keywords_list": "keyword"})
    .drop(pl.col("authors"), pl.col("keywords"))
)

# identify columns that are json formatted in order to clean to be comma separated
json_array_cols = [
    col
    for col, dtype in df_parsed.schema.items()
    if dtype == pl.String
    and df_parsed.select(pl.col(col).drop_nulls().str.starts_with("[").all()).item()
]

# clean up commas in json columns and marking blank string as null. Converting string date to date type
df_final = (
    df_parsed.with_columns(
        [
            pl.col(col).str.json_decode(pl.List(pl.String)).list.join(", ").alias(col)
            for col in json_array_cols
        ]
    )
)

In [46]:
df_final.describe()

statistic,pmid,doi,article_type,article_title,article_subject,pub_date,reference_count,journal_title,publisher_name,copyright_statement,copyright_year,abstract,affiliations,has_supplemental,figure_count,table_count,funding,data_available_details,data_available,code_available_details,code_available,given_names,is_corresponding,orcid,surname,keyword,author_full_name
str,str,str,str,str,str,str,f64,str,str,str,f64,str,str,f64,f64,f64,str,str,f64,str,f64,str,f64,str,str,str,str
"""count""","""397097""","""396732""","""397097""","""397097""","""395174""","""396450""",397097.0,"""397097""","""363216""","""382503""",343169.0,"""392171""","""397097""",397097.0,397097.0,397097.0,"""397097""","""397097""",397097.0,"""397097""",397097.0,"""397079""",397079.0,"""397079""","""397079""","""375495""","""397079"""
"""null_count""","""0""","""365""","""0""","""0""","""1923""","""647""",0.0,"""0""","""33881""","""14594""",53928.0,"""4926""","""0""",0.0,0.0,0.0,"""0""","""0""",0.0,"""0""",0.0,"""18""",18.0,"""18""","""18""","""21602""","""18"""
"""mean""",null,null,null,null,null,"""2025-06-25 16:02:22.746878""",57.717266,null,null,null,2024.948745,null,null,0.702458,3.895595,3.405274,null,null,0.359413,null,0.034183,null,0.04959,null,null,null,null
"""std""",null,null,null,null,null,null,80.153558,null,null,null,0.242374,null,null,null,3.294737,4.435322,null,null,null,null,null,null,null,null,null,null,null
"""min""","""38168309""","""10.1001/jamahealthforum.2025.1…","""abstract""","""10 Practical Considerations fo…","""10""","""2024-03-20""",0.0,"""aace endocrinology and diabete…","""a k peters/crc press""","""(c) 2025 The authors; licensee…",2023.0,"""(1) Background: Alveolar bone …","""""",0.0,0.0,0.0,"""""","""""",0.0,"""""",0.0,"""""",0.0,"""""","""""",""" borrelia burgdorferi sensu …",""" """
"""25%""",null,null,null,null,null,"""2025-03-27""",32.0,null,null,null,2025.0,null,null,null,2.0,1.0,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,"""2025-07-02""",43.0,null,null,null,2025.0,null,null,null,3.0,3.0,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,"""2025-09-30""",61.0,null,null,null,2025.0,null,null,null,5.0,4.0,null,null,null,null,null,null,null,null,null,null,null
"""max""","""41716853""","""10.9758/cpn.24.1261""","""systematic-review""","""대형 언어 모델을 활용한 연구를 위한 TRIPOD-LL…","""Zoster""","""2026-05-12""",1194.0,"""yonsei medical journal""","""zenodo""","""©Zongqi Xia, Prerna Chikersal,…",2026.0,"""​​BackgroundOsteoporosis, a pr…","""‡Department of Neurosurgery, V…",1.0,487.0,77.0,"""贵州省第八批高层次创新型“百”层次人才""","""“For ethical and administrativ…",1.0,"""eHarmonize is an open-source p…",1.0,"""“Andy”""",1.0,"""https://orcid.org/0009-0009-99…","""Țânțu""",""" propofol""","""“Andy” Ho Wing Chan"""


In [47]:
cols_with_null_info = [
    (col, df_final.select(pl.col(f"{col}")).null_count().item(), df_final.schema[col])
    for col in df_final.columns
    if df_final.select(pl.col(col).null_count()).item() > 0
]

cols_with_null_info

[('doi', 365, String),
 ('article_subject', 1923, String),
 ('pub_date', 647, Date),
 ('publisher_name', 33881, String),
 ('copyright_statement', 14594, String),
 ('copyright_year', 53928, Int16),
 ('abstract', 4926, String),
 ('given_names', 18, String),
 ('is_corresponding', 18, Boolean),
 ('orcid', 18, String),
 ('surname', 18, String),
 ('keyword', 21602, String),
 ('author_full_name', 18, String)]

## Analysis

In [48]:
df_final.schema

Schema([('pmid', String),
        ('doi', String),
        ('article_type', String),
        ('article_title', String),
        ('article_subject', String),
        ('pub_date', Date),
        ('reference_count', Int32),
        ('journal_title', String),
        ('publisher_name', String),
        ('copyright_statement', String),
        ('copyright_year', Int16),
        ('abstract', String),
        ('affiliations', String),
        ('has_supplemental', Boolean),
        ('figure_count', Int16),
        ('table_count', Int16),
        ('funding', String),
        ('data_available_details', String),
        ('data_available', Boolean),
        ('code_available_details', String),
        ('code_available', Boolean),
        ('given_names', String),
        ('is_corresponding', Boolean),
        ('orcid', String),
        ('surname', String),
        ('keyword', String),
        ('author_full_name', String)])

In [49]:
df.describe()

statistic,pmid,doi,article_type,article_title,article_subject,authors,pub_date,keywords,reference_count,license_type,journal_title,publisher_name,copyright_statement,copyright_year,abstract,affiliations,has_supplemental,figure_count,table_count,funding,data_available_details,data_available,code_available_details,code_available
str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,f64,str,str,str,str,f64,str,f64
"""count""","""7178""","""7167""","""7178""","""7178""","""7178""","""7178""","""7178""","""7178""",7178.0,"""0""","""7178""","""7178""","""7178""","""7178""","""7178""","""7178""",7178.0,"""7178""","""7178""","""7178""","""7178""",7178.0,"""7178""",7178.0
"""null_count""","""0""","""11""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""7178""","""0""","""0""","""0""","""0""","""0""","""0""",0.0,"""0""","""0""","""0""","""0""",0.0,"""0""",0.0
"""mean""",null,null,null,null,null,null,null,null,53.387852,null,null,null,null,null,null,null,0.657147,null,null,null,null,0.395653,null,0.033853
"""std""",null,null,null,null,null,null,null,null,48.141303,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""min""","""38168309""","""10.1001/jamahealthforum.2025.1…","""abstract""",""" Helicobacter pylori Screeni…","""""","""[]""","""2024-10-08""","""["" Artificial intelligence"",""S…",0.0,null,"""AACE Endocrinology and Diabete…","""""","""""","""""","""""","""[""*Brain Trauma Foundation, Pa…",0.0,"""0""","""0""","""["" The APC was funded by Wenta…","""["" Data are available on reque…",0.0,"""[""A GitHub repository containi…",0.0
"""25%""",null,null,null,null,null,null,null,null,31.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""50%""",null,null,null,null,null,null,null,null,43.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""75%""",null,null,null,null,null,null,null,null,59.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""max""","""41716853""","""10.9758/cpn.24.1261""","""systematic-review""","""대형 언어 모델을 활용한 연구를 위한 TRIPOD-LL…","""Zoster""","""[{""given_names"":""Šimon"",""is_co…","""2026-5-12""","""[]""",1194.0,null,"""npj Metabolic Health and Disea…","""ecancer Global Foundation""","""©Zongqi Xia, Prerna Chikersal,…","""2026""","""​​BackgroundOsteoporosis, a pr…","""[]""",1.0,"""9""","""9""","""[]""","""[]""",1.0,"""[]""",1.0


In [50]:
# creating a summary dataframe of common metrics
summary_df = pl.DataFrame(
    {
        "Metric": [
            "Total Articles",
            "Unique Authors",
            "Unique Keywords",
            "Avg Authors per Paper",
            "Avg Keywords per Paper",
        ],
        "Value": [
            df_final.n_unique("pmid"),
            df_final.n_unique("author_full_name"),
            df_final.n_unique("keyword"),
            df_final.group_by("pmid")
            .agg(pl.col("author_full_name").n_unique().alias("n_authors"))
            .select(pl.col("n_authors").mean().round(2))
            .item(),
            df_final.group_by("pmid")
            .agg(pl.col("keyword").n_unique().alias("n_keywords"))
            .select(pl.col("n_keywords").mean().round(2))
            .item(),
        ],
    },
    strict=False,
)

summary_df

Metric,Value
str,f64
"""Total Articles""",7178.0
"""Unique Authors""",72964.0
"""Unique Keywords""",16177.0
"""Avg Authors per Paper""",11.47
"""Avg Keywords per Paper""",4.99


#### Building Visuals

In [51]:
data_avail = df_final.unique(subset=["pmid"]).group_by("data_available").agg(pl.len().alias("count"))

data_avail

data_available,count
bool,u32
false,4338
true,2840


In [52]:
pub_year_counts = (
    df_final.with_columns(pl.col("pub_date").dt.year().alias("pub_year"))
    .group_by("pub_year")
    .agg(pl.col("pmid").n_unique().alias("num_articles"))
    .sort("pub_year")
)

pub_year_counts

pub_year,num_articles
i32,u32
null,22
2024,91
2025,7058
2026,7


In [53]:
pub_month_counts = (
    df_final
    .with_columns(
        pl.col("pub_date")
          .dt.truncate("1mo")
          .alias("pub_month")
    )
    .group_by("pub_month")
    .agg(pl.col("pmid").n_unique().alias("num_articles"))
    .sort("pub_month")
)

pub_month_counts

pub_month,num_articles
date,u32
null,22
2024-03-01,1
2024-04-01,1
2024-05-01,2
2024-07-01,7
…,…
2025-11-01,643
2025-12-01,566
2026-01-01,4


In [54]:
top_keywords = (
    df_final
    .filter(pl.col("keyword").is_not_null())
    .group_by("keyword")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

top_keywords

keyword,count
str,u32
"""machine learning""",4341
"""artificial intelligence""",3640
"""epidemiology""",2283
"""covid-19""",2171
"""mortality""",1667
…,…
"""prognosis""",1519
"""computed tomography""",1472
"""antibiotics""",1404


In [55]:
top_authors = (
    df_final
    .filter(pl.col("author_full_name").is_not_null())
    .group_by("author_full_name")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .head(15)
)

top_authors

author_full_name,count
str,u32
""" """,1591
"""Kiyohide Fushimi""",205
"""Hideo Yasunaga""",110
"""Hiroki Matsui""",97
"""Jiang Bian""",92
…,…
"""Wei Zhang""",62
"""Gabor Erdoes""",62
"""Yang Liu""",62


#### Visualizing

In [56]:
(
    alt.Chart(data_avail)
    .mark_bar()
    .encode(
        x=alt.X("data_available:O", title=""),
        y=alt.Y("count:Q", title="Number of Articles"),
    )
    .properties(title="Data Availability")
)

alt.Chart(...)

In [57]:
(
    alt.Chart(pub_year_counts)
    .mark_bar()
    .encode(
        x=alt.X("pub_year:O", title="Publication Year"),
        y=alt.Y("num_articles:Q", title="Number of Articles"),
    )
    .properties(title="Publications per Year")
)

alt.Chart(...)

In [58]:
(
    alt.Chart(pub_month_counts)
    .mark_line(point=True)
    .encode(
        x=alt.X("pub_month:T", title="Publication Date"),
        y=alt.Y("num_articles:Q", title="Number of Articles")
    )
    .properties(title="Publications Over Time")
)

alt.Chart(...)

In [59]:
(
    alt.Chart(top_keywords)
    .mark_bar()
    .encode(
        x=alt.X(
            "keyword:O",
            title="Keyword",
            sort=alt.SortField("count", order="descending"),
        ),
        y=alt.Y("count:Q", title="Count"),
    )
    .properties(title="Top 15 Keywords")
)

alt.Chart(...)

In [60]:
(
    alt.Chart(top_authors)
    .mark_bar()
    .encode(
        x=alt.X(
            "author_full_name:O",
            title="Author",
            sort=alt.SortField("count", order="descending"),
        ),
        y=alt.Y("count:Q", title="Number of Articles"),
    )
    .properties(title="Top 15 Authors")
)

alt.Chart(...)